# MedGraph — Fine-tuning QLoRA sobre Llama-3.2-3B-Instruct

**Tech Challenge Fase 3 · 8IADT**
Alexandre Carneiro do Carmo (RM370980) · Brunno Costa Castigrini (RM371429) ·
Pedro Henrique Azevedo Aragão (RM373481) · Valter Willian de Oliveira Filho (RM370979)

---

## O que este notebook faz

Ajusta um modelo `Llama-3.2-3B-Instruct` para a tarefa do assistente clínico do
Hospital Vida Plena, usando **QLoRA** — o modelo base carregado em 4 bits, com
adaptadores de baixo posto treinados por cima.

## Por que no Colab, e não na máquina local

O projeto inteiro roda em um MacBook com Apple Silicon, exceto esta etapa. Treinar
3 bilhões de parâmetros exige uma GPU com CUDA: na T4 gratuita do Colab o treino
leva cerca de 1 hora; no Apple Silicon levaria várias, sem `bitsandbytes` para a
quantização em 4 bits.

O artefato que sai daqui — um adapter de ~50 MB — volta para a máquina local, é
fundido ao modelo base, convertido para GGUF e servido pelo Ollama.

## Por que QLoRA, e não fine-tuning completo

| | Fine-tuning completo | QLoRA |
|---|---|---|
| VRAM necessária (3B) | ~48 GB | **~9 GB** |
| Parâmetros treinados | 3.2 bilhões | **~24 milhões** (0,7%) |
| Artefato gerado | ~6 GB | **~50 MB** |
| Cabe na T4 gratuita | não | **sim** |

Os 50 MB do adapter cabem no repositório Git, o que significa que o resultado do
treino é versionado junto com o código que o produziu.

---

## Antes de começar

1. **Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU**
2. Aceite a licença do Llama 3.2 em https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct
   (gratuito, aprovação imediata)
3. Crie um token em https://huggingface.co/settings/tokens com permissão de **escrita**

Tempo total estimado: **60 a 90 minutos**.

## 1. Verificação do ambiente

Falhar aqui custa segundos. Descobrir a ausência de GPU depois do download do modelo custa vinte minutos.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

## 2. Instalação das dependências

As versões não são fixadas em valores exatos de propósito. O Colab atualiza sua
imagem base com frequência, e um `==` que funcionava há seis meses costuma
conflitar com o `torch` pré-instalado hoje. Fixamos apenas os limites inferiores
que garantem as APIs que usamos, e o notebook imprime as versões efetivamente
instaladas logo em seguida — se algo quebrar, a tabela é o ponto de partida do
diagnóstico.

⚠️ **Reinicie a sessão depois desta célula** (Ambiente de execução → Reiniciar sessão)
e continue a partir da célula 3.

In [ ]:
# Uma linha só, de propósito: continuação com barra invertida dentro de um
# magic do IPython é ambígua entre versões, e um erro aqui só apareceria na
# primeira execução do notebook — que é justamente quando não se quer depurar.
%pip install -q -U "transformers>=4.46" "trl>=0.12" "peft>=0.13" "bitsandbytes>=0.44" "accelerate>=1.0" "datasets>=3.0" "huggingface_hub>=0.26"

print("\nInstalação concluída. REINICIE A SESSÃO antes de continuar.")

## 3. Clonagem do repositório

O dataset de fine-tuning é versionado no repositório. Isso torna o treino reproduzível: quem clonar obtém exatamente os mesmos exemplos, na mesma ordem.

In [ ]:
import os, sys

REPO = "https://github.com/alexandreccarmo/fia_tech3.git"
if not os.path.isdir("/content/fia_tech3"):
    !git clone --depth 1 {REPO} /content/fia_tech3

os.chdir("/content/fia_tech3")
sys.path.insert(0, "/content/fia_tech3/src")
sys.path.insert(0, "/content/fia_tech3")

!ls -lh data/processed/sft_*.jsonl

## 4. Ambiente de GPU e versões

A tabela de versões vai para o cartão de treino gravado junto ao adapter — é o que permite reconstruir este ambiente meses depois.

In [ ]:
from medgraph.finetune import colab_utils

gpu = colab_utils.verificar_gpu()
versoes = colab_utils.versoes_instaladas()

print("GPU")
for chave, valor in gpu.items():
    print(f"  {chave:.<18} {valor}")

print("\nVersões")
for pacote, versao in versoes.items():
    print(f"  {pacote:.<18} {versao}")

if not gpu["suporta_bf16"]:
    print("\nGPU sem suporte a bfloat16 (esperado na T4). O treino usará float16.")

## 5. Autenticação no Hugging Face

Necessária porque o Llama 3.2 é um modelo *gated*: exige aceite de licença e token.

In [ ]:
from huggingface_hub import login, whoami

# Cole o token quando solicitado. Ele NÃO fica salvo no notebook.
login()
print("Autenticado como:", whoami()["name"])

## 6. Carregamento do dataset

Cada exemplo é uma conversa de três turnos:

- **system** — o prompt que define quem é o MedGraph e quais são seus limites
- **user** — a pergunta com o contexto recuperado, já no formato `[E1]`, `[P1]`, `[C1]`
- **assistant** — a resposta de referência, citando as fontes

O prompt de sistema é **o mesmo** usado na inferência (vem de
`src/medgraph/chains/prompts.py`). Se treino e inferência divergissem nesse ponto,
o modelo abandonaria o formato de citação em produção — justamente o que sustenta
o requisito de *explainability*.

In [ ]:
from datasets import load_dataset

dados = load_dataset(
    "json",
    data_files={
        "train": "data/processed/sft_train.jsonl",
        "validation": "data/processed/sft_valid.jsonl",
    },
)
print(dados)

exemplo = dados["train"][0]
for mensagem in exemplo["messages"]:
    print(f"\n{'='*72}\n[{mensagem['role'].upper()}]\n{'='*72}")
    print(mensagem["content"][:900])

## 7. Modelo base em 4 bits

`NF4` (NormalFloat4) é o tipo de dado proposto no artigo do QLoRA: otimizado para
pesos que seguem distribuição normal — o caso dos pesos de uma rede treinada.
Perde menos qualidade que o int4 comum.

A **dupla quantização** comprime também as constantes de quantização, economizando
cerca de 0,4 bit por parâmetro. Com 16 GB de VRAM, essa margem importa.

Alternativa sem *gating*: troque `MODELO_BASE` por `Qwen/Qwen2.5-3B-Instruct`.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# =============================================================================
# ESCOLHA DO MODELO BASE
# =============================================================================
# O Llama 3.2 é *gated*: exige aceite de licença, e o pedido pode ficar pendente.
# Se a célula falhar com `GatedRepoError` ou 403, troque para o Qwen — que é
# aberto, do mesmo tamanho e com suporte a português comparável.
#
# A troca é segura: os módulos-alvo do LoRA (q,k,v,o,gate,up,down_proj) são
# idênticos nas duas arquiteturas, e o Modelfile do Ollama não fixa template —
# usa o que vem embutido no GGUF. Nada mais no projeto precisa mudar.
#
# O enunciado do Tech Challenge pede "um modelo LLM (como LLaMA, Falcon ou um
# outro)" — a escolha é explicitamente livre.
#
# ATENÇÃO: use o MESMO valor no notebook 02_exportar_gguf.
# =============================================================================

MODELO_BASE = "meta-llama/Llama-3.2-3B-Instruct"   # gated — exige aceite
# MODELO_BASE = "Qwen/Qwen2.5-3B-Instruct"         # aberto — descomente se o de cima falhar


tokenizador = AutoTokenizer.from_pretrained(MODELO_BASE)
if tokenizador.pad_token is None:
    # Llama não define token de preenchimento. Reaproveitar o de fim de
    # sequência é a convenção; o rótulo do preenchimento é mascarado no
    # cálculo da perda, então isso não contamina o treino.
    tokenizador.pad_token = tokenizador.eos_token
tokenizador.padding_side = "right"

modelo = AutoModelForCausalLM.from_pretrained(
    MODELO_BASE,
    quantization_config=colab_utils.config_quantizacao(gpu["suporta_bf16"]),
    device_map="auto",
    dtype=torch.bfloat16 if gpu["suporta_bf16"] else torch.float16,
)
modelo.config.use_cache = False           # incompatível com gradient checkpointing
modelo.config.pretraining_tp = 1

print(f"Parâmetros: {modelo.num_parameters()/1e9:.2f} B")
print(f"VRAM ocupada: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

## 8. Configuração do LoRA

Adaptadores aplicados às **sete projeções** — atenção (`q`, `k`, `v`, `o`)
e MLP (`gate`, `up`, `down`). Aplicar apenas em `q_proj`/`v_proj`, como é comum em
tutoriais, rende menos quando a tarefa muda o **estilo** da resposta — e é o nosso
caso: queremos que o modelo passe a responder num formato rígido, com decisão na
primeira linha e citação no fim.

⚠️ Esta célula **prepara** o modelo, mas não cria os adaptadores. Quem os cria é o
`SFTTrainer`, na célula seguinte. Criar aqui também faria o LoRA ser aplicado duas
vezes — ver o comentário no código.

In [ ]:
from peft import prepare_model_for_kbit_training

lora = colab_utils.config_lora()
print("Configuração LoRA:")
for chave in ("r", "lora_alpha", "lora_dropout", "target_modules"):
    print(f"  {chave:.<18} {getattr(lora, chave)}")

modelo = prepare_model_for_kbit_training(modelo, use_gradient_checkpointing=True)

# =============================================================================
# NÃO chame get_peft_model aqui.
# =============================================================================
# Uma versão anterior desta célula chamava get_peft_model(modelo, lora) apenas
# para imprimir quantos parâmetros seriam treinados, e depois apagava o wrapper
# com `del`. Isso NÃO desfaz nada: get_peft_model modifica o próprio `modelo`,
# anexando a ele um atributo peft_config.
#
# A célula seguinte passa peft_config ao SFTTrainer, que então aplica LoRA uma
# SEGUNDA vez, sobre a primeira. O resultado são dois adaptadores empilhados,
# com dtypes inconsistentes — e o treino morre com
#
#     NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda"
#     not implemented for 'BFloat16'
#
# porque o GradScaler do fp16 recebe gradientes em bf16. O erro aparece no
# treinador, longe da célula que o causou, e a mensagem não menciona PEFT.
#
# O PEFT avisa: "You are trying to modify a model with PEFT for a second time".
# É um UserWarning no meio da saída da célula 9 — fácil de não ver.
# =============================================================================

total = sum(p.numel() for p in modelo.parameters())
print(f"\nParâmetros do modelo base: {total / 1e9:.2f} B")
print("Os adaptadores LoRA serão criados pelo SFTTrainer, na próxima célula.")

## 9. Treinador

A função `montar_configuracao_sft` inspeciona a assinatura da versão instalada do `trl` e passa apenas os argumentos aceitos, imprimindo os que foram ignorados. A biblioteca renomeou parâmetros várias vezes em pouco tempo; sem essa camada, o notebook quebraria a cada atualização do Colab.

In [ ]:
# =============================================================================
# ONDE OS CHECKPOINTS SÃO GRAVADOS
# =============================================================================
# /content é o disco EFÊMERO da máquina do Colab. Se a sessão apenas se
# desconectar, os arquivos sobrevivem e o treino pode ser retomado. Se a VM for
# reciclada — o que acontece após desconexões longas ou ao esgotar a cota —,
# /content é apagado e o treino se perde inteiro.
#
# Montar o Google Drive protege contra o segundo caso, ao custo de cerca de um
# minuto de configuração e de ~1,5 GB do seu espaço (dois checkpoints).
#
# Recomendado se você estiver treinando em horário de pico ou já tiver perdido
# uma sessão hoje.
# =============================================================================

USAR_DRIVE = False   # mude para True para sobreviver à reciclagem da VM

if USAR_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    SAIDA = "/content/drive/MyDrive/medgraph/saida_treino"
else:
    SAIDA = "/content/saida_treino"

print(f"Checkpoints em: {SAIDA}")
if not USAR_DRIVE:
    print("AVISO: disco efêmero. Se a VM for reciclada, o treino se perde.")

configuracao = colab_utils.montar_configuracao_sft(SAIDA)

treinador = colab_utils.montar_treinador(
    modelo=modelo,
    tokenizador=tokenizador,
    dados_treino=dados["train"],
    dados_validacao=dados["validation"],
    configuracao=configuracao,
    lora=lora,
)

passos = (
    len(dados["train"])
    * colab_utils.CONFIG_PADRAO["num_train_epochs"]
    // (colab_utils.CONFIG_PADRAO["per_device_train_batch_size"]
        * colab_utils.CONFIG_PADRAO["gradient_accumulation_steps"])
)
print(f"\nExemplos de treino .... {len(dados['train']):,}")
print(f"Exemplos de validação . {len(dados['validation']):,}")
print(f"Passos de otimização .. ~{passos}")

## 10. Treino

⏱️ **Medido: ~45 segundos por passo numa T4.** A célula anterior imprimiu quantos
passos serão dados — multiplique. Com a configuração padrão são 484 passos, ou
seja **quase 6 horas**, mais do que a cota diária do Colab gratuito. Veja
*Treinar em menos tempo* no guia (`docs/guia_colab.md`) antes de começar.

**Mantenha esta aba aberta** — o Colab gratuito desconecta sessões ociosas.

Checkpoints são gravados a cada `save_steps` passos (padrão: 100) no diretório
que a célula 9 imprimiu — `/content/saida_treino`, ou uma pasta do seu Drive se
você ligou `USAR_DRIVE`.

**Se a conexão cair, não reinicie a sessão por reflexo.** Perder o navegador não
é perder o treino: a VM continua executando. Recarregue a página primeiro, o que
reconecta a interface ao mesmo kernel sem interromper nada. Só se o kernel tiver
mesmo morrido é que se refaz o caminho — células 3 a 9 e, então,

```python
resultado = treinador.train(resume_from_checkpoint=True)
```

In [ ]:
import time

inicio = time.time()
resultado = treinador.train()
duracao = time.time() - inicio

print(f"\nTreino concluído em {duracao/60:.1f} min")
print(f"Perda final de treino: {resultado.training_loss:.4f}")

## 11. Curva de perda

O gráfico responde à pergunta "o treino funcionou?" sem ambiguidade: perda de treino caindo com a de validação subindo indica sobreajuste; ambas paradas indicam taxa de aprendizado baixa demais ou dados insuficientes.

In [ ]:
from IPython.display import Image, display

historico = treinador.state.log_history
caminho_grafico = colab_utils.grafico_de_perda(historico, "docs/graficos/curva_de_perda.png")
display(Image(caminho_grafico))

perdas_eval = [h["eval_loss"] for h in historico if "eval_loss" in h]
if perdas_eval:
    print(f"Perda de validação: {perdas_eval[0]:.4f} -> {perdas_eval[-1]:.4f} "
          f"(redução de {(1 - perdas_eval[-1]/perdas_eval[0]):.1%})")

## 12. Verificação qualitativa

Antes de exportar, uma checagem de sanidade: o modelo aprendeu o **formato**?

Três coisas precisam aparecer na resposta:
1. a primeira linha começando com `Decisão:` e um dos três rótulos;
2. uma justificativa ancorada na evidência fornecida;
3. a linha `Fontes:` com o marcador `[E1]`.

Se o formato não aparecer, a avaliação da Etapa 4 não conseguirá extrair o rótulo
e todo o comparativo fica comprometido.

In [ ]:
import json, re

def gerar(mensagens, max_novos=220):
    entrada = tokenizador.apply_chat_template(
        mensagens, tokenize=False, add_generation_prompt=True
    )
    tokens = tokenizador(entrada, return_tensors="pt").to(modelo.device)
    with torch.no_grad():
        saida = modelo.generate(
            **tokens,
            max_new_tokens=max_novos,
            do_sample=False,
            pad_token_id=tokenizador.pad_token_id,
        )
    return tokenizador.decode(saida[0][tokens["input_ids"].shape[1]:], skip_special_tokens=True)


with open("data/processed/pubmedqa_teste.jsonl", encoding="utf-8") as arquivo:
    teste = [json.loads(linha) for linha in arquivo if linha.strip()]

from medgraph.chains import prompts

acertos_formato = 0
for caso in teste[:5]:
    mensagens = [
        {"role": "system", "content": prompts.SISTEMA},
        {"role": "user", "content": prompts.usuario_decisao(caso["pergunta"], caso["contexto"])},
    ]
    resposta = gerar(mensagens)
    tem_decisao = bool(re.match(r"Decis[ãa]o:\s*(yes|no|maybe)", resposta.strip(), re.I))
    tem_fonte = "[E1]" in resposta
    acertos_formato += tem_decisao and tem_fonte

    print(f"\n{'='*72}")
    print(f"Pergunta: {caso['pergunta'][:110]}")
    print(f"Esperado: {caso['decisao']}")
    print(f"{'-'*72}\n{resposta.strip()}")
    print(f"{'-'*72}")
    print(f"formato: decisão={'ok' if tem_decisao else 'FALHOU'} | fonte={'ok' if tem_fonte else 'FALHOU'}")

print(f"\n\nFormato correto em {acertos_formato}/5 exemplos.")
if acertos_formato < 4:
    print("ATENÇÃO: o modelo não aderiu ao formato. Considere mais uma época de treino.")

## 13. Gravação do adapter e do cartão de treino

O cartão de treino é o que impede o adapter de virar um binário sem procedência: registra modelo base, dados, hiperparâmetros, GPU, versões, duração e perdas.

In [ ]:
DESTINO = "models/adapters/medgraph-llama32-3b-lora"

treinador.model.save_pretrained(DESTINO)
tokenizador.save_pretrained(DESTINO)

# A curva de perda entra TAMBÉM no diretório do adapter, e não só em
# docs/graficos/. O zip da célula seguinte leva apenas este diretório, e
# /content é efêmero: sem isto, o gráfico que o roteiro do vídeo manda
# mostrar ficaria preso no Colab. Redesenhamos do histórico em vez de
# copiar o arquivo, para não depender de a célula 11 ter sido executada.
colab_utils.grafico_de_perda(historico, f"{DESTINO}/curva_de_perda.png")

colab_utils.salvar_metadados(
    f"{DESTINO}/cartao_de_treino.json",
    modelo_base=MODELO_BASE,
    configuracao=colab_utils.CONFIG_PADRAO,
    versoes=versoes,
    gpu=gpu,
    historico=historico,
    exemplos_treino=len(dados["train"]),
    exemplos_validacao=len(dados["validation"]),
    duracao_s=duracao,
)

!du -sh {DESTINO}
!ls -la {DESTINO}

## 14. Download do adapter

Baixe o arquivo `.zip` e descompacte em `models/adapters/` no repositório local.
São ~50 MB — cabe no Git, e é assim que o resultado do treino fica versionado
junto com o código que o produziu.

O passo seguinte é o notebook **`02_exportar_gguf.ipynb`**, que funde o adapter ao
modelo base, converte para GGUF quantizado e publica no Hugging Face Hub — de onde
a máquina local o baixa para servir pelo Ollama.

In [ ]:
from google.colab import files

!cd models/adapters && zip -qr /content/medgraph-adapter.zip medgraph-llama32-3b-lora
!ls -lh /content/medgraph-adapter.zip

files.download("/content/medgraph-adapter.zip")

---

## O que fazer com o resultado

| Passo | Onde | Comando |
|---|---|---|
| 1. Descompactar o adapter | máquina local | `unzip medgraph-adapter.zip -d models/adapters/` |
| 2. Exportar para GGUF | Colab | notebook `02_exportar_gguf.ipynb` |
| 3. Registrar no Ollama | máquina local | `make modelo` |
| 4. Avaliar | máquina local | `make avaliar` |

## Se algo der errado

| Sintoma | Causa provável | O que fazer |
|---|---|---|
| `CUDA out of memory` | sequência longa demais | `colab_utils.CONFIG_PADRAO["max_seq_length"] = 768` antes da célula 9 |
| `401` ao carregar o modelo | licença do Llama não aceita | aceite em huggingface.co e refaça o login |
| Sessão desconectou | ociosidade do Colab gratuito | `treinador.train(resume_from_checkpoint=True)` |
| `TypeError` no `SFTTrainer` | mudança de API do `trl` | leia a lista de argumentos ignorados impressa na célula 9 |
| Formato incorreto na célula 12 | treino insuficiente | aumente `num_train_epochs` para 3 |